# PRISM Rebuttal - Phase 3: PEFT / LoRA

Addresses Reviewer fWEj weakness 1 and question 1, who wrote:
*"Adding a small-scale PEFT experiment can show if the observations still
hold when the models are adapted to the downstream task. My score would
increase if this analysis were provided."*

**Design.** Deliberately narrow, so it can be completed and reported honestly
rather than gestured at.

| axis | choice | why |
|---|---|---|
| models | UNI (307M), H-Optimus-0 (1.1B) | both plain timm ViTs, so LoRA targets are unambiguous; both belong to the model group our headline OOD claim covers; 3.6x size gap tests whether adaptation closes the reliability gap |
| datasets | MHIST (binary, 3.1k), LungHist700 (7-class, 691) | the two datasets where degeneracy and calibration inversion appear; both small enough to fine-tune repeatedly |
| fractions | 1%, 10%, 100% | the endpoints plus one interior point |
| seeds | 42, 123, 456 | matches the frozen-probe protocol |

36 fine-tuning runs total.

**Protocol parity.** Identical stratified subsets, identical seeds, identical
metrics, temperature fitted on the same held-out validation split. The only
thing that changes is the probe: LoRA-adapted backbone plus linear head
instead of a linear probe on frozen features.

**Preprocessing.** Uses `timm.data.resolve_model_data_config`, i.e. each
model's own documented transform. This is what the paper claimed and did not
do; this experiment does it, and we say so in the rebuttal.

Requires a GPU. Checkpointed per (model, dataset) so it can be resumed.

In [1]:
# torchao Colab da 0.10 olarak geliyor, yeni transformers >=0.16 istiyor.
# transformers torchao yu bulamazsa kontrolu atliyor, o yuzden kaldiriyoruz.
!pip uninstall -y -q torchao
!pip install -q peft timm

import os, gc, glob, time, warnings
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from PIL import Image
import timm
from timm.data import resolve_model_data_config, create_transform
from peft import LoraConfig, get_peft_model
from google.colab import drive, userdata
from huggingface_hub import login

warnings.filterwarnings('ignore')
drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE    = '/content/drive/MyDrive/PRISM'
DATA    = f'{BASE}/datasets'
OUT_DIR = f'{BASE}/results_v2'
CKPT    = f'{OUT_DIR}/peft_parts'
os.makedirs(CKPT, exist_ok=True)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
FRACTIONS  = [0.01, 0.10, 1.00]
SEEDS      = [42, 123, 456]
N_BINS     = 15
EPOCHS     = 10
LR         = 1e-4
BATCH      = 32
LORA_R     = 8
LORA_ALPHA = 16

try:
    import torchao
    print('UYARI: torchao hala kurulu', torchao.__version__)
except ImportError:
    print('torchao yok, sorun cikarmayacak')

print(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'GPU YOK',
      '| bos GB:',
      round(torch.cuda.mem_get_info()[0] / 1e9, 1) if DEVICE == 'cuda' else '-')

Mounted at /content/drive
torchao yok, sorun cikarmayacak
NVIDIA A100-SXM4-80GB | bos GB: 84.6


## 1. Shared metrics (identical to the frozen-probe protocol)

In [2]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_correct(proba, y):
    if proba.shape[1] == 2:
        return proba[:, 1], (y == 1).astype(float)
    return proba.max(1), (proba.argmax(1) == y).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def ece_adaptive(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    e = np.quantile(c, np.linspace(0, 1, n_bins + 1))
    e[0], e[-1] = 0.0, 1.0 + 1e-9
    e = np.unique(e)
    return ece_fixed(proba, y, n_bins) if len(e) < 3 else _ece_edges(c, k, e)

def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(val_logits, val_y, bounds=(0.1, 10.0)):
    idx = np.arange(len(val_y))
    def nll(T):
        p = softmax_np(val_logits / T)
        return float(-np.log(p[idx, val_y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked = []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        picked.extend(np.random.choice(ci, size=max(1, int(len(ci) * fraction)),
                                       replace=False))
    return np.array(sorted(picked))

def degeneracy(pred, k):
    cnt = np.bincount(pred, minlength=k)
    return float(cnt.max() / cnt.sum())

print('metrics ready')

metrics ready


## 2. Datasets

Splits must match the frozen-probe experiments exactly, otherwise the
comparison is meaningless. MHIST uses its official partition column with a
15% validation split carved from train using `random_state=42`, which is what
the embedding notebooks did. LungHist700 uses the same 70/15/15 stratified
split with `random_state=42`.

In [3]:
MHIST_DIR = f'{DATA}/mhist'
LUNG_DIR  = f'{DATA}/lunghist700'
Image.MAX_IMAGE_PIXELS = None


class ListDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
        self.labels = np.array([l for _, l in samples])
    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return self.transform(Image.open(p).convert('RGB')), l


def mhist_splits():
    import csv
    tr, te = [], []
    with open(f'{MHIST_DIR}/annotations.csv') as f:
        for row in csv.DictReader(f):
            p = f"{MHIST_DIR}/images/{row['Image Name']}"
            y = 0 if row['Majority Vote Label'] == 'HP' else 1
            (te if row['Partition'].lower() == 'test' else tr).append((p, y))
    df = pd.DataFrame(tr, columns=['p', 'y'])
    val = df.sample(frac=0.15, random_state=42)
    trn = df.drop(val.index)
    return (list(trn.itertuples(index=False, name=None)),
            list(val.itertuples(index=False, name=None)), te)


def lunghist_splits():
    from sklearn.model_selection import train_test_split
    samples = []
    for root, _, files in os.walk(LUNG_DIR):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')):
                samples.append((os.path.join(root, f),
                                os.path.basename(root)))
    classes = sorted({c for _, c in samples})
    c2i = {c: i for i, c in enumerate(classes)}
    samples = [(p, c2i[c]) for p, c in samples]
    y = [l for _, l in samples]
    tr, tmp = train_test_split(samples, train_size=0.70, random_state=42,
                               stratify=y)
    y2 = [l for _, l in tmp]
    va, te = train_test_split(tmp, train_size=0.50, random_state=42,
                              stratify=y2)
    return tr, va, te


SPLITS = {'mhist': mhist_splits, 'lunghist700': lunghist_splits}

for name, fn in SPLITS.items():
    tr, va, te = fn()
    print(f'{name:<12} train={len(tr):>5}  val={len(va):>4}  test={len(te):>5}  '
          f'classes={len(set(l for _, l in tr))}')

mhist        train= 1849  val= 326  test=  977  classes=2
lunghist700  train=  483  val= 104  test=  104  classes=7


## 3. LoRA models

LoRA is applied to the fused attention projections (`qkv`) and the attention
output projection (`proj`), the standard targets for timm ViTs. The backbone
is otherwise frozen; the classification head is trained in full.

In [4]:
MODEL_SPECS = {
    'UNI':         dict(hub='hf-hub:MahmoodLab/uni',
                        kwargs=dict(init_values=1e-5, dynamic_img_size=True)),
    'H-Optimus-0': dict(hub='hf-hub:bioptimus/H-optimus-0',
                        kwargs=dict(init_values=1e-5, dynamic_img_size=False)),
}


class LoRAClassifier(nn.Module):
    def __init__(self, model_name, n_classes):
        super().__init__()
        spec = MODEL_SPECS[model_name]
        backbone = timm.create_model(spec['hub'], pretrained=True,
                                     num_classes=0, **spec['kwargs'])
        self.data_config = resolve_model_data_config(backbone)
        dim = backbone.num_features

        cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                         target_modules=['qkv', 'proj'], bias='none')
        self.backbone = get_peft_model(backbone, cfg)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

    def trainable_report(self):
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        a = sum(p.numel() for p in self.parameters())
        return t, a, 100 * t / a


def transforms_for(model_name):
    # each model's own documented preprocessing, via timm
    spec = MODEL_SPECS[model_name]
    m = timm.create_model(spec['hub'], pretrained=False, num_classes=0,
                          **spec['kwargs'])
    cfg = resolve_model_data_config(m)
    del m; gc.collect()
    return (create_transform(**cfg, is_training=True),
            create_transform(**cfg, is_training=False))


for name in MODEL_SPECS:
    tr_t, ev_t = transforms_for(name)
    print(f'{name}: eval transform = {ev_t}')

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

UNI: eval transform = Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=[224, 224])
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


config.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

H-Optimus-0: eval transform = Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=[224, 224])
    MaybeToTensor()
    Normalize(mean=tensor([0.7072, 0.5787, 0.7036]), std=tensor([0.2119, 0.2301, 0.1775]))
)


## 4. Train and evaluate one configuration

In [5]:
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE == 'cuda'):
            out = model(x.to(DEVICE, non_blocking=True))
        L.append(out.float().cpu().numpy())
        Y.append(np.asarray(y))
    return np.vstack(L), np.concatenate(Y)


def run_config(model_name, dataset, fraction, seed):
    tr, va, te = SPLITS[dataset]()
    n_classes = len(set(l for _, l in tr))
    tr_t, ev_t = transforms_for(model_name)

    y_tr = np.array([l for _, l in tr])
    idx = stratified_sample(y_tr, fraction, seed)
    tr_sub = [tr[i] for i in idx]

    ds_tr = ListDataset(tr_sub, tr_t)
    ds_va = ListDataset(va, ev_t)
    ds_te = ListDataset(te, ev_t)

    g = torch.Generator(); g.manual_seed(seed)
    dl_tr = DataLoader(ds_tr, batch_size=min(BATCH, len(ds_tr)), shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=False,
                       generator=g)
    dl_va = DataLoader(ds_va, batch_size=64, num_workers=2, pin_memory=True)
    dl_te = DataLoader(ds_te, batch_size=64, num_workers=2, pin_memory=True)

    torch.manual_seed(seed)
    model = LoRAClassifier(model_name, n_classes).to(DEVICE)
    n_train_p, n_all_p, pct = model.trainable_report()

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, EPOCHS * len(dl_tr)))
    lossf = nn.CrossEntropyLoss()

    t0 = time.time()
    for _ in range(EPOCHS):
        model.train()
        for x, y in dl_tr:
            x = x.to(DEVICE, non_blocking=True)
            y = torch.as_tensor(y).to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.bfloat16,
                                enabled=DEVICE == 'cuda'):
                loss = lossf(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
    train_s = time.time() - t0

    Lva, Yva = collect_logits(model, dl_va)
    Lte, Yte = collect_logits(model, dl_te)

    proba = softmax_np(Lte)
    pred  = proba.argmax(1)

    try:
        auroc = (roc_auc_score(Yte, proba[:, 1]) if n_classes == 2
                 else roc_auc_score(Yte, proba, multi_class='ovr',
                                    average='macro'))
    except Exception:
        auroc = np.nan

    T = fit_temperature(Lva, Yva)
    scaled = softmax_np(Lte / T)

    row = dict(
        probe='lora', model=model_name, dataset=dataset,
        fraction=fraction, seed=seed, n_train=len(idx), n_classes=n_classes,
        trainable_params=n_train_p, trainable_pct=round(pct, 3),
        epochs=EPOCHS, lr=LR, lora_r=LORA_R, train_seconds=round(train_s, 1),
        auroc=auroc,
        f1_macro=f1_score(Yte, pred, average='macro', zero_division=0),
        brier=(brier_score_loss(Yte, proba[:, 1]) if n_classes == 2 else np.nan),
        ece_fixed=ece_fixed(proba, Yte),
        ece_adaptive=ece_adaptive(proba, Yte),
        temperature=T,
        ece_scaled_fixed=ece_fixed(scaled, Yte),
        ece_scaled_adaptive=ece_adaptive(scaled, Yte),
        degeneracy_share=degeneracy(pred, n_classes),
    )
    row['degenerate'] = row['degeneracy_share'] > 0.99

    del model, opt, dl_tr, dl_va, dl_te
    gc.collect(); torch.cuda.empty_cache()
    return row


print('run_config ready')

run_config ready


## 5. Run everything (checkpointed)

In [6]:
GRID = [(m, d) for d in ['lunghist700', 'mhist']
                for m in ['UNI', 'H-Optimus-0']]

for model_name, dataset in GRID:
    out = f'{CKPT}/{model_name.replace("-","_")}__{dataset}.csv'
    if os.path.exists(out):
        print(f'  skip (done): {model_name} x {dataset}')
        continue
    rows = []
    for frac in FRACTIONS:
        for seed in SEEDS:
            try:
                r = run_config(model_name, dataset, frac, seed)
                rows.append(r)
                print(f'{model_name:>12} {dataset:<12} f={frac:<5} s={seed} '
                      f'auroc={r["auroc"]:.4f} ece={r["ece_fixed"]:.4f} '
                      f'T={r["temperature"]:.2f} '
                      f'{"[DEGEN]" if r["degenerate"] else ""} '
                      f'({r["train_seconds"]:.0f}s)')
            except Exception as e:
                print(f'{model_name:>12} {dataset:<12} f={frac} s={seed} '
                      f'FAILED: {type(e).__name__}: {e}')
                gc.collect(); torch.cuda.empty_cache()
    if rows:
        pd.DataFrame(rows).to_csv(out, index=False)
        print(f'  -> saved {out}')

parts = sorted(glob.glob(f'{CKPT}/*.csv'))
df_peft = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_peft.to_csv(f'{OUT_DIR}/peft_all.csv', index=False)
print(f'\n{len(parts)}/4 cells complete, {len(df_peft)} runs -> peft_all.csv')

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.21GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

         UNI lunghist700  f=0.01  s=42 auroc=0.5095 ece=0.1003 T=5.38  (17s)
         UNI lunghist700  f=0.01  s=123 auroc=0.5321 ece=0.1266 T=10.00  (12s)
         UNI lunghist700  f=0.01  s=456 auroc=0.5768 ece=0.1089 T=1.77  (10s)
         UNI lunghist700  f=0.1   s=42 auroc=0.6032 ece=0.0541 T=1.30  (39s)
         UNI lunghist700  f=0.1   s=123 auroc=0.5860 ece=0.0643 T=4.05  (40s)
         UNI lunghist700  f=0.1   s=456 auroc=0.6352 ece=0.0577 T=0.99  (31s)
         UNI lunghist700  f=1.0   s=42 auroc=0.9602 ece=0.1957 T=0.63  (279s)
         UNI lunghist700  f=1.0   s=123 auroc=0.9675 ece=0.1856 T=0.55  (98s)
         UNI lunghist700  f=1.0   s=456 auroc=0.9559 ece=0.2024 T=0.58  (98s)
  -> saved /content/drive/MyDrive/PRISM/results_v2/peft_parts/UNI__lunghist700.csv


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 4.54GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

 H-Optimus-0 lunghist700  f=0.01  s=42 auroc=0.6027 ece=0.0337 T=1.28  (7s)
 H-Optimus-0 lunghist700  f=0.01  s=123 auroc=0.4819 ece=0.1373 T=2.71  (7s)
 H-Optimus-0 lunghist700  f=0.01  s=456 auroc=0.5385 ece=0.1535 T=10.00  (7s)
 H-Optimus-0 lunghist700  f=0.1   s=42 auroc=0.6598 ece=0.0575 T=0.71  (21s)
 H-Optimus-0 lunghist700  f=0.1   s=123 auroc=0.5773 ece=0.0526 T=0.79  (21s)
 H-Optimus-0 lunghist700  f=0.1   s=456 auroc=0.6131 ece=0.0616 T=1.72  (21s)
 H-Optimus-0 lunghist700  f=1.0   s=42 auroc=0.9559 ece=0.1638 T=0.56  (112s)
 H-Optimus-0 lunghist700  f=1.0   s=123 auroc=0.9591 ece=0.1883 T=0.52  (112s)
 H-Optimus-0 lunghist700  f=1.0   s=456 auroc=0.9635 ece=0.2424 T=0.54  (112s)
  -> saved /content/drive/MyDrive/PRISM/results_v2/peft_parts/H_Optimus_0__lunghist700.csv
         UNI mhist        f=0.01  s=42 auroc=0.7236 ece=0.0556 T=0.71  (22s)
         UNI mhist        f=0.01  s=123 auroc=0.4445 ece=0.1407 T=2.08 [DEGEN] (19s)
         UNI mhist        f=0.01  s=456 auroc=0

## 6. LoRA versus frozen linear probe

The question fWEj asked: do the paper's observations survive adaptation?

In [7]:
frozen = pd.read_csv(f'{OUT_DIR}/indomain_all_v2.csv')
frozen = frozen[frozen.model.isin(['UNI', 'H-Optimus-0']) &
                frozen.dataset.isin(['MHIST', 'LungHist700'])].copy()
frozen['dataset'] = frozen['dataset'].str.lower()
frozen['probe'] = 'linear'

cols = ['probe','model','dataset','fraction','auroc','f1_macro',
        'ece_fixed','ece_scaled_fixed','temperature','degeneracy_share']
both = pd.concat([frozen[cols], df_peft[cols]], ignore_index=True)
agg = both.groupby(['dataset','model','fraction','probe']).mean(numeric_only=True)

for metric in ['auroc', 'ece_fixed', 'ece_scaled_fixed', 'f1_macro']:
    print(f'\n=== {metric} : linear vs lora ===')
    t = agg[metric].unstack('probe')
    t['delta'] = t['lora'] - t['linear']
    print(t.round(4).to_string())

print('\n=== Does adaptation remove the degenerate collapse? ===')
d = agg['degeneracy_share'].unstack('probe').round(3)
print(d.to_string())

print('\n=== Does the calibration inversion on LungHist700 survive? ===')
for m in ['UNI', 'H-Optimus-0']:
    for probe in ['linear', 'lora']:
        s = agg.xs(('lunghist700', m), level=('dataset','model'))
        v = s.xs(probe, level='probe')['ece_fixed'].sort_index()
        if len(v) >= 2:
            trend = 'WORSENS' if v.iloc[-1] > v.iloc[0] else 'improves'
            print(f'  {m:>12} {probe:>7}: '
                  f'{v.iloc[0]:.4f} -> {v.iloc[-1]:.4f}  ECE {trend} with labels')

both.to_csv(f'{OUT_DIR}/peft_vs_linear.csv', index=False)
print('\nSaved -> peft_vs_linear.csv')


=== auroc : linear vs lora ===
probe                             linear    lora   delta
dataset     model       fraction                        
lunghist700 H-Optimus-0 0.01      0.6407  0.5410 -0.0997
                        0.05      0.7540     NaN     NaN
                        0.10      0.8247  0.6167 -0.2080
                        0.25      0.9207     NaN     NaN
                        0.50      0.9409     NaN     NaN
                        1.00      0.9638  0.9595 -0.0043
            UNI         0.01      0.6790  0.5394 -0.1396
                        0.05      0.8111     NaN     NaN
                        0.10      0.8683  0.6081 -0.2602
                        0.25      0.9267     NaN     NaN
                        0.50      0.9447     NaN     NaN
                        1.00      0.9634  0.9612 -0.0022
mhist       H-Optimus-0 0.01      0.7012  0.5265 -0.1747
                        0.05      0.7680     NaN     NaN
                        0.10      0.8119  0.7624 -0.0496

## 7. What to report

Three outcomes are possible and all three are reportable.

**Observations survive adaptation.** Strongest result: the decoupling and the
calibration inversion are properties of the representations, not artefacts of
the linear-probe protocol. Report directly.

**Observations weaken under LoRA.** Also useful, and honest: it bounds the
scope of the claim to frozen-feature deployment, which is still the dominant
setting in computational pathology because it needs no GPU at adaptation
time. State the bound explicitly.

**LoRA collapses at 1% labels.** Expected, and itself a finding: adaptation
does not rescue the low-label regime, it needs data too.

Do not report only whichever outcome is convenient. Reviewer tp5b reads
code.

In [1]:
import pandas as pd, numpy as np, glob, os

OUT_DIR = '/content/drive/MyDrive/PRISM/results_v2'

parts = sorted(glob.glob(f'{OUT_DIR}/peft_parts/*.csv'))
print(f'tamamlanan hucre: {len(parts)}/4')
for p in parts:
    print('  ', os.path.basename(p))

peft = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
peft.to_csv(f'{OUT_DIR}/peft_all.csv', index=False)
print(f'\n{len(peft)} kosum -> peft_all.csv')
print('basarisiz:', peft['auroc'].isna().sum())

# donmus probe temel cizgisi
lin = pd.read_csv(f'{OUT_DIR}/indomain_all_v2.csv')
lin = lin[lin.model.isin(['UNI','H-Optimus-0']) &
          lin.dataset.isin(['MHIST','LungHist700'])].copy()
lin['dataset'] = lin['dataset'].str.lower()
lin['probe'] = 'linear'
peft['probe'] = 'lora'

C = ['probe','model','dataset','fraction','auroc','f1_macro',
     'ece_fixed','ece_scaled_fixed','temperature','degeneracy_share']
both = pd.concat([lin[C], peft[C]], ignore_index=True)
both.to_csv(f'{OUT_DIR}/peft_vs_linear.csv', index=False)
agg = both.groupby(['dataset','model','fraction','probe']).mean(numeric_only=True)

for m in ['auroc','ece_fixed','ece_scaled_fixed','f1_macro','degeneracy_share']:
    t = agg[m].unstack('probe')
    if 'lora' not in t.columns:
        continue
    t['delta'] = t['lora'] - t['linear']
    print(f'\n=== {m} ===')
    print(t.round(4).to_string())

print('\n=== LungHist700 kalibrasyon tersinmesi ===')
for mdl in ['UNI','H-Optimus-0']:
    for pr in ['linear','lora']:
        try:
            v = agg.xs((('lunghist700'), mdl), level=('dataset','model')) \
                   .xs(pr, level='probe')['ece_fixed'].sort_index()
            print(f'  {mdl:>12} {pr:>7}: {v.iloc[0]:.4f} -> {v.iloc[-1]:.4f}  '
                  f'{"KOTULESIYOR" if v.iloc[-1] > v.iloc[0] else "iyilesiyor"}')
        except Exception as e:
            print(f'  {mdl:>12} {pr:>7}: yok')

print('\n=== sicaklik ===')
print(agg['temperature'].unstack('probe').round(3).to_string())

print('\n=== egitilebilir parametre ===')
print(peft.groupby('model')[['trainable_params','trainable_pct',
                             'train_seconds']].mean().round(2).to_string())

tamamlanan hucre: 0/4


ValueError: No objects to concatenate